# GPU Telemetry Classifier

Classifies GPU telemetry traces as **training** or **not training** using three physically-grounded threshold rules:

1. **Power** — sustained high power (>400W) indicates heavy compute
2. **Tensor/SM ratio** — high ratio (>0.25) indicates matmul-dominated workload
3. **NVLink autocorrelation** — periodic NVLink bursts (peak >0.3) indicate allreduce heartbeat

A 60-second window is flagged if **any** rule triggers.

In [1]:
import sys
sys.path.insert(0, '..')

from classifier.features import load_telemetry, extract_features
from classifier.rules import classify_windows, overall_verdict, Thresholds
from classifier.display import show_results

print('Classifier loaded.')

Classifier loaded.


---
## Classify a trace

Set `CSV_PATH` to any telemetry CSV, then run the next cell.

In [20]:
CSV_PATH = '../data/t4_telemetry.csv'

In [21]:
df = load_telemetry(CSV_PATH)
features = extract_features(df)
verdicts = classify_windows(features)
show_results(verdicts)

Window,Verdict,Power,Tensor/SM,NVLink AC,Triggered
0–60s,FLAG,356W,0.404,0.698,"tensor_ratio, nvlink_autocorr"
60–120s,FLAG,360W,0.410,0.664,"tensor_ratio, nvlink_autocorr"
120–180s,FLAG,363W,0.419,0.692,"tensor_ratio, nvlink_autocorr"
180–240s,FLAG,363W,0.425,0.690,"tensor_ratio, nvlink_autocorr"
240–300s,FLAG,239W,0.200,0.921,nvlink_autocorr
300–360s,CLEAN,117W,0.000,0.000,-
360–420s,CLEAN,115W,0.000,0.000,-
420–480s,CLEAN,115W,0.000,0.000,-
480–540s,CLEAN,114W,0.000,0.000,-
540–600s,CLEAN,114W,0.000,0.000,-


---
## Validation — all 11 conditions

Runs the classifier against every collected dataset and shows a summary.

In [ ]:
from pathlib import Path
from IPython.display import display, HTML

DATA_DIR = Path('../data')

CONDITIONS = {
    'T1': ('t1_telemetry.csv', 'Large DDP (3.37B)',       True),
    'T2': ('t2_telemetry.csv', 'Small DDP (136M)',        True),
    'T3': ('t3_telemetry.csv', 'Grad Accum (16x)',        True),
    'T4': ('t4_telemetry.csv', 'Pipeline Parallel',       True),
    'T5': ('t5_telemetry.csv', 'Grad Checkpoint (6.7B)',  True),
    'T6': ('t6_telemetry.csv', 'FSDP / ZeRO-3',          True),
    'E3': ('e3_telemetry.csv', 'Intermittent Training',   True),
    'E4': ('e4_telemetry.csv', 'PCIe-Only Allreduce',     True),
    'I2': ('i2_telemetry.csv', 'Streaming Inference',     False),
    'I3': ('i3_telemetry.csv', 'vLLM High-Throughput',    False),
    'B1': ('b1_telemetry.csv', 'Idle + Model Loaded',     False),
}

rows_html = ''
correct = 0
total = 0

for code, (fname, label, expected_training) in CONDITIONS.items():
    path = DATA_DIR / fname
    if not path.exists():
        continue
    total += 1

    df = load_telemetry(str(path))
    feats = extract_features(df)
    vds = classify_windows(feats)
    detected, n_flagged, n_total = overall_verdict(vds)

    # Which rules triggered across all windows
    all_rules = set()
    for v in vds:
        all_rules.update(v.triggered_rules)
    rules_str = ', '.join(sorted(all_rules)) if all_rules else '-'

    is_correct = detected == expected_training
    if is_correct:
        correct += 1

    expected_str = 'Training' if expected_training else 'Clean'
    actual_str = 'Training' if detected else 'Clean'

    if is_correct and detected:
        badge_bg, badge_color = '#dcfce7', '#15803d'  # green — true positive
        badge = 'TP'
    elif is_correct and not detected:
        badge_bg, badge_color = '#dcfce7', '#15803d'  # green — true negative
        badge = 'TN'
    elif detected and not expected_training:
        badge_bg, badge_color = '#fee2e2', '#b91c1c'  # red — false positive
        badge = 'FP'
    else:
        badge_bg, badge_color = '#fee2e2', '#b91c1c'  # red — false negative
        badge = 'FN'

    rows_html += f'''
    <tr style="border-bottom:1px solid #e5e7eb;">
        <td style="padding:6px 12px; font-family:monospace; font-weight:bold;">{code}</td>
        <td style="padding:6px 12px;">{label}</td>
        <td style="padding:6px 12px; text-align:center;">{expected_str}</td>
        <td style="padding:6px 12px; text-align:center;">{actual_str}</td>
        <td style="padding:6px 12px; text-align:center;">{n_flagged}/{n_total}</td>
        <td style="padding:6px 12px; text-align:center;">
            <span style="background:{badge_bg}; color:{badge_color}; padding:2px 8px;
                         border-radius:4px; font-weight:bold; font-size:12px;">{badge}</span>
        </td>
        <td style="padding:6px 12px; font-family:monospace; color:#666;">{rules_str}</td>
    </tr>'''

acc_color = '#15803d' if correct == total else '#b91c1c'

html = f'''
<div style="font-family:monospace; font-size:16px; font-weight:bold; margin:8px 0 12px 0;
            color:{acc_color};">
    Accuracy: {correct}/{total} ({100*correct/total:.0f}%)
</div>
<table style="border-collapse:collapse; width:100%; font-size:14px;">
    <thead>
        <tr style="border-bottom:2px solid #999;">
            <th style="padding:6px 12px; text-align:left;">Code</th>
            <th style="padding:6px 12px; text-align:left;">Condition</th>
            <th style="padding:6px 12px; text-align:center;">Expected</th>
            <th style="padding:6px 12px; text-align:center;">Predicted</th>
            <th style="padding:6px 12px; text-align:center;">Flagged</th>
            <th style="padding:6px 12px; text-align:center;">Result</th>
            <th style="padding:6px 12px; text-align:left;">Rules</th>
        </tr>
    </thead>
    <tbody>{rows_html}</tbody>
</table>
'''
display(HTML(html))

---
## Threshold sweep

How much margin does each threshold have? Sweeps one threshold at a time while holding the other two fixed at defaults.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Pre-extract features for all conditions
all_feats = {}
for code, (fname, label, expected) in CONDITIONS.items():
    path = DATA_DIR / fname
    if not path.exists():
        continue
    df = load_telemetry(str(path))
    all_feats[code] = (extract_features(df), expected)


def sweep_accuracy(param_name, values):
    """Sweep one threshold, return per-value accuracy."""
    accuracies = []
    for val in values:
        kwargs = {
            'power': Thresholds().power,
            'tensor_ratio': Thresholds().tensor_ratio,
            'nvlink_autocorr': Thresholds().nvlink_autocorr,
        }
        kwargs[param_name] = val
        t = Thresholds(**kwargs)
        n_correct = 0
        n_total = 0
        for code, (feats, expected) in all_feats.items():
            vds = classify_windows(feats, t)
            detected, _, _ = overall_verdict(vds)
            if detected == expected:
                n_correct += 1
            n_total += 1
        accuracies.append(n_correct / n_total)
    return accuracies


sweeps = {
    'power': (np.arange(50, 750, 10), Thresholds().power, 'Power threshold (W)'),
    'tensor_ratio': (np.arange(0.0, 0.8, 0.01), Thresholds().tensor_ratio, 'Tensor/SM ratio threshold'),
    'nvlink_autocorr': (np.arange(0.0, 1.0, 0.02), Thresholds().nvlink_autocorr, 'NVLink autocorr threshold'),
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (param, (values, default, xlabel)) in zip(axes, sweeps.items()):
    accs = sweep_accuracy(param, values)
    ax.plot(values, accs, 'k-', linewidth=1.5)
    ax.axvline(x=default, color='#b91c1c', linewidth=1.5, linestyle='--', alpha=0.7,
               label=f'Current: {default}')

    # Shade the region where accuracy is 100%
    perfect = [v for v, a in zip(values, accs) if a == 1.0]
    if perfect:
        ax.axvspan(min(perfect), max(perfect), alpha=0.1, color='#15803d',
                   label=f'100% range: {min(perfect):.1f}–{max(perfect):.1f}')

    ax.set_xlabel(xlabel)
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Threshold Sensitivity — Single-Threshold Sweep (others held fixed)', fontsize=14)
fig.tight_layout()
fig.savefig(Path('../plots/12_threshold_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()